In [ ]:
# MUSK evaluation performance summary notebook
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
from collections import defaultdict
print('Creating MUSK performance summary...')

In [ ]:
# Load all MUSK evaluation results
all_results = []

# Process all input files
for result_file in snakemake.input:
    try:
        with open(result_file, 'r') as f:
            # MUSK results are in JSON Lines format
            for line in f:
                if line.strip():
                    data = json.loads(line)
                    # Add file path info for context
                    data['result_file'] = str(result_file)
                    all_results.append(data)
                    
    except Exception as e:
        print(f'Error loading {result_file}: {e}')

print(f'Loaded {len(all_results)} MUSK evaluation results')

In [ ]:
# Convert results to DataFrame for analysis
if all_results:
    # Create a structured DataFrame
    structured_results = []
    
    for result in all_results:
        assert result.get('model') == "spotwhisperer_local"
        base_info = {
            'dataset': result.get('dataset'),
            # 'model': result.get('model'),
            'pretrained': result.get('pretrained'),
            'task': result.get('task'),
            'language': result.get('language', 'en')
        }
        
        # Flatten metrics
        if 'metrics' in result:
            for metric_name, metric_value in result['metrics'].items():
                row = base_info.copy()
                row['metric_name'] = metric_name
                row['metric_value'] = metric_value
                structured_results.append(row)
    
    df = pd.DataFrame(structured_results)
    print(f'\nStructured results shape: {df.shape}')
    print(df.head())
else:
    print('No results found - creating empty summary')
    df = pd.DataFrame()

In [ ]:
# Create summary tables by task
summary_by_task = {}

for task in df['task'].unique():
    task_df = df[df['task'] == task]
    
    # Pivot to get metrics as columns
    pivot_df = task_df.pivot_table(
        index='dataset', # 'model', 
        columns='metric_name', 
        values='metric_value',
        aggfunc='mean'  # In case of duplicates
    )
    
    summary_by_task[task] = pivot_df
    print(f'\n=== {task.upper()} RESULTS ===')
    print(pivot_df)
    
# Save detailed summary
summary_dict = {
    'task_summaries': {task: summary.to_dict() for task, summary in summary_by_task.items()},
    'overall_summary': {
        'total_evaluations': len(all_results),
        'unique_datasets': df['dataset'].nunique() if not df.empty else 0,
        'unique_tasks': df['task'].nunique() if not df.empty else 0,
        # 'models_evaluated': df['model'].unique().tolist() if not df.empty else []
    }
}

In [ ]:
# Create visualizations

plt.style.use('seaborn-v0_8')

# Plot 1: Performance by task and dataset
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('MUSK Evaluation Results Summary', fontsize=16)

# Subplot 1: Classification tasks accuracy
classification_metrics = df[df['metric_name'].str.contains('acc', case=False, na=False)]
if not classification_metrics.empty:
    sns.barplot(data=classification_metrics, x='dataset', y='metric_value', 
               hue='task', ax=axes[0,0])
    axes[0,0].set_title('Classification Accuracy by Dataset')
    axes[0,0].set_ylabel('Accuracy')
    axes[0,0].tick_params(axis='x', rotation=45)

# Subplot 2: Retrieval tasks recall
retrieval_metrics = df[df['metric_name'].str.contains('recall', case=False, na=False)]
if not retrieval_metrics.empty:
    sns.barplot(data=retrieval_metrics, x='dataset', y='metric_value', 
               hue='metric_name', ax=axes[0,1])
    axes[0,1].set_title('Retrieval Recall by Dataset')
    axes[0,1].set_ylabel('Recall')
    axes[0,1].tick_params(axis='x', rotation=45)

# Subplot 3: Task distribution
task_counts = df['task'].value_counts()
axes[1,0].pie(task_counts.values, labels=task_counts.index, autopct='%1.1f%%')
axes[1,0].set_title('Distribution of Evaluation Tasks')

# Subplot 4: Performance summary table
axes[1,1].axis('tight')
axes[1,1].axis('off')

# Create summary table for display
if 'zeroshot_classification' in summary_by_task:
    cls_summary = summary_by_task['zeroshot_classification']
    table_data = cls_summary.round(3).reset_index()
    table = axes[1,1].table(cellText=table_data.values, colLabels=table_data.columns,
                           cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    axes[1,1].set_title('Zero-shot Classification Summary')

plt.tight_layout()
plt.show()

In [ ]:
# Save the summary
output_path = Path(snakemake.output.summary)
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w') as f:
    json.dump(summary_dict, f, indent=2)

print(f'\nSaved MUSK evaluation summary to: {output_path}')
print(f'Total evaluations processed: {summary_dict["overall_summary"]["total_evaluations"]}')
print(f'Tasks completed: {summary_dict["overall_summary"]["unique_tasks"]}')
print(f'Datasets evaluated: {summary_dict["overall_summary"]["unique_datasets"]}')